### Restricting to Non-Finnish Europeans to estimate mutation rate scaling and residual variance

In [1]:
# --- make parent folder importable ---
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))  # points to Paper_SFS/

# --- usual imports ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy as sc
from scipy.integrate import quad
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from scipy.signal import find_peaks
from scipy.stats import gaussian_kde
from scipy.stats import gmean, linregress, norm, beta, uniform, lognorm
from scipy.integrate import simpson, trapezoid
from scipy.optimize import minimize
from scipy.special import logsumexp
from scipy.integrate import simpson
from scipy import stats
import glob
import os
import math
import dask
import dask.dataframe as dd
import re
import dask.bag as db
import csv
from tqdm import tqdm
from dask.diagnostics import ProgressBar
from concurrent.futures import ProcessPoolExecutor
import time
import joblib
from joblib import Parallel, delayed, parallel_backend
import multiprocessing
import pickle
import random

# --- import your project modules (files in Paper_SFS/) ---
import Gen_SFS_with_s
import plot_SFS

# (optional) pull specific functions
# from Gen_SFS_with_s import calc_SFS_k
# from plot_SFS import plot_sfs


In [2]:
def load_syn_anc(demography, data_root="../Data/syn_anc"):
    """
    Load synonymous ancestry-specific data for a given demography.

    Parameters
    ----------
    demography : str - e.g. 'NFE', 'AFR', 'EAS'
    data_root : str or Path - Path to Data/syn_anc directory

    Returns
    -------
    df : pandas.DataFrame
    allele_number : allele_number mode for that demography
    """
    data_root = Path(data_root)

    matches = sorted(data_root.glob(f"{demography}_*.txt.gz"))

    if len(matches) == 0:
        raise FileNotFoundError(f"No files found for demography '{demography}' in {data_root}")

    if len(matches) > 1:
        raise ValueError(f"Multiple files found for demography '{demography}':\n" + "\n".join(str(p.name) for p in matches))

    file_path = matches[0]

    # Extract the number e.g. "24590" from "NFE_24590.txt.gz"
    filename = os.path.basename(file_path)
    num_str = filename.split("_")[1].split(".")[0]   # split on "_" then remove ".txt.gz"
    allele_number = int(num_str)

    # Load the file
    df = pd.read_csv(file_path, sep="\t", compression="gzip")

    return df, allele_number

### Choose demography

In [3]:
demography = "NFE"

In [4]:
# Load the QC'ed NFE synonymous dataset- Contains Roulette rate (MR), allele count (allele_count), number of points corresponding 
# to unique roulette rate and allele count (count)

df, AN = load_syn_anc(demography, data_root = "../Data/syn_anc")
df_syn_filtered = df[df['allele_count']<=5000]
df_syn_filtered = df_syn_filtered.copy()
df_syn_filtered['log_MR'] = np.log(df_syn_filtered['MR'])
df_syn_filtered

,MR,allele_count,count,log_MR
0,0.004,0.0,72,-5.521461
1,0.004,1.0,1,-5.521461
2,0.013,0.0,852944,-4.342806
3,0.013,1.0,16265,-4.342806
4,0.013,2.0,3579,-4.342806
...,...,...,...,...
75694,3.912,523.0,1,1.364049
75695,3.912,639.0,1,1.364049
75696,3.912,843.0,1,1.364049
75697,3.912,1155.0,1,1.364049


In [5]:
# Count of unique allele counts for each MR [SFS(k) - how many different k's are present for each MR]
MR_counts = df_syn_filtered['MR'].value_counts().sort_index()

## 3 parameter model of mutation rate distribution
Based on Seplyarskiy et al. (2021, Science), we model the mutation rate distribution ($\mu$) as p($\mu$) ~ Gamma($\mu$|shape = $\alpha$, rate = $\lambda$) * (1-p) + $\delta(\mu-\mu_{low})$ * p. \
We choose $\mu_{low}$ = $2\times10^{-10}$

In [6]:
mu_low = 2e-10
mu, s, n, SFS_low_mu = Gen_SFS_with_s.compute_SFS(mu = mu_low, s = 0, n = AN) # Generate the low mu SFS

In [7]:
def log_likelihood_3_param(params, df_syn, SFS_low_mu, n, demography = "schraiber_et_al", kmax = 5000):

    """
    Computes the negative log-likelihood for given values of mu
    - `params`: mean and var of mu and weight p
    - `df_syn`: DataFrame with allele_count and SFS counts (count)
    """
    mean_mu, var_mu, p = params

    # Compute log(mu_scaled) only once
    _, _, _, SFS_neutral = Gen_SFS_with_s.compute_SFS_gamma_mu_var(mu = mean_mu, var = var_mu, s = 0, n = n, kmax=kmax, demography=demography)

    # Get k values as integer array
    k_indices = df_syn["allele_count"].astype(int).values

    # Get counts values as integer array
    counts = df_syn["count"].astype(int).values

    # Compute likelihood
    log_likelihood =  np.log((1-p)*SFS_neutral[k_indices]+p*SFS_low_mu[k_indices] + 1e-300) * counts

    logL = np.sum(log_likelihood)  # Sum log-likelihood across all loci

    return -logL  # Return negative log-likelihood for minimization


In [8]:
# Define a function for parallel optimization
def optimize_for_MR(i):
    MR = MR_filtered[i]
    df_syn_analyze = df_syn_filtered[df_syn_filtered['MR'] == MR]
    initial_params = [MR * 5e-8 * 4 * 14448, MR * 1e-15 * (4 * 14448) ** 2, 0.0001]
    bounds = [(2e-10 * 4 * 14448, 3e-6 * 4 * 14448), (1e-23 * (4 * 14448) ** 2, 1e-13 * (4 * 14448) ** 2), (0, 0.2)]

    result = minimize(
        log_likelihood_3_param,
        initial_params,
        args=(df_syn_analyze, SFS_low_mu, AN),
        bounds=bounds,
        method="Nelder-Mead",
        options={"disp": True, "fatol": 1e-6}
    )

    mean_mu = result.x[0]
    var_mu = result.x[1]
    p_mu = result.x[2]
    loglik = -result.fun

    return i, MR, p_mu, mean_mu, var_mu, loglik

# Run in parallel and write results
output_dir = Path("param_mut_rate")
output_dir.mkdir(exist_ok=True)

output_file = output_dir / f"results_mu_var_p_{demography}.txt"

# Filter MR values with more than 0 points
MR_filtered = MR_counts[MR_counts > 0].index # Basically unique MR values which are non NA

with open(output_file, "w") as f:
    f.write("MR\tp_mu\tmean_mu\tvar_mu\tmax_loglikelihood\n")

    with ProcessPoolExecutor() as executor:
        for i, MR, p_mu, mean_mu, var_mu, loglik in executor.map(optimize_for_MR, range(1,len(MR_filtered))):  # we lose the first roulette value MR = 0.004 which has 2 points
            f.write(f"{MR}\t{p_mu}\t{mean_mu}\t{var_mu}\t{loglik}\n")


Optimization terminated successfully.
         Current function value: 810046.677756
         Iterations: 109
         Function evaluations: 255
Optimization terminated successfully.
         Current function value: 827840.316209
         Iterations: 115
         Function evaluations: 268
Optimization terminated successfully.
         Current function value: 842807.318386
         Iterations: 123
         Function evaluations: 275
Optimization terminated successfully.
         Current function value: 785543.015529
         Iterations: 115
         Function evaluations: 316
Optimization terminated successfully.
         Current function value: 413030.983328
         Iterations: 124
         Function evaluations: 320
Optimization terminated successfully.
         Current function value: 671669.636024
         Iterations: 132
         Function evaluations: 324
Optimization terminated successfully.
         Current function value: 319818.507275
         Iterations: 123
         Function ev

/tmp/ipykernel_3588671/636012983.py:8: RuntimeWarning: Maximum number of function evaluations has been exceeded.
  result = minimize(


Optimization terminated successfully.
         Current function value: 3819.377238
         Iterations: 56
         Function evaluations: 104
Optimization terminated successfully.
         Current function value: 4059.913204
         Iterations: 79
         Function evaluations: 136
Optimization terminated successfully.
         Current function value: 4595.097853
         Iterations: 52
         Function evaluations: 95
Optimization terminated successfully.
         Current function value: 3871.553089
         Iterations: 93
         Function evaluations: 164
Optimization terminated successfully.
         Current function value: 4116.399561
         Iterations: 96
         Function evaluations: 173
Optimization terminated successfully.
         Current function value: 4220.964308
         Iterations: 46
         Function evaluations: 89
Optimization terminated successfully.
         Current function value: 4042.351336
         Iterations: 73
         Function evaluations: 126
Optimiza

/tmp/ipykernel_3588671/636012983.py:8: RuntimeWarning: Maximum number of function evaluations has been exceeded.
  result = minimize(


Optimization terminated successfully.
         Current function value: 36728.288158
         Iterations: 157
         Function evaluations: 276
Optimization terminated successfully.
         Current function value: 90012.668560
         Iterations: 136
         Function evaluations: 244
Optimization terminated successfully.
         Current function value: 127106.225968
         Iterations: 126
         Function evaluations: 228
Optimization terminated successfully.
         Current function value: 113514.372875
         Iterations: 148
         Function evaluations: 267
Optimization terminated successfully.
         Current function value: 59780.835475
         Iterations: 179
         Function evaluations: 319
Optimization terminated successfully.
         Current function value: 146436.022187
         Iterations: 123
         Function evaluations: 213
Optimization terminated successfully.
         Current function value: 2535.492724
         Iterations: 89
         Function evaluati

## 2 parameter model - Mutation rate mean and variance

We model the mutation rate distribution ($\mu$) as p($\mu$) ~ Gamma($\mu$|shape = $\alpha$, rate = $\lambda$) 

In [9]:
def log_likelihood_2_param(params, df_syn, n, demography = "schraiber_et_al", kmax = 5000):

    """
    Computes the negative log-likelihood for given values of mu
    - `params`: mean and var of mu
    - `df_syn`: DataFrame with allele_count and SFS counts (count)
    """
    mean_mu, var_mu = params

    # Compute log(mu_scaled) only once
    _, _, _, SFS_neutral = Gen_SFS_with_s.compute_SFS_gamma_mu_var(mu = mean_mu, var = var_mu, s = 0, n = n, kmax=kmax, demography=demography)

    # Get k values as integer array
    k_indices = df_syn["allele_count"].astype(int).values

    # Get counts values as integer array
    counts = df_syn["count"].astype(int).values

    # Compute likelihood
    log_likelihood =  np.log(SFS_neutral[k_indices] + 1e-300) * counts

    logL = np.sum(log_likelihood)  # Sum log-likelihood across all loci

    return -logL  # Return negative log-likelihood for minimization


In [10]:
# Define a function for parallel optimization
def optimize_for_MR(i):
    MR = MR_filtered[i]
    df_syn_analyze = df_syn_filtered[df_syn_filtered['MR'] == MR]
    initial_params = [MR * 5e-8 * 4 * 14448, MR * 1e-15 * (4 * 14448) ** 2]
    bounds = [(2e-10 * 4 * 14448, 3e-6 * 4 * 14448), (1e-23 * (4 * 14448) ** 2, 1e-13 * (4 * 14448) ** 2)]

    result = minimize(
        log_likelihood_2_param,
        initial_params,
        args=(df_syn_analyze, AN),
        bounds=bounds,
        method="Nelder-Mead",
        options={"disp": True, "fatol": 1e-6}
    )

    mean_mu = result.x[0]
    var_mu = result.x[1]
    loglik = -result.fun

    return i, MR, mean_mu, var_mu, loglik

# Run in parallel and write results
output_dir = Path("param_mut_rate")
output_dir.mkdir(exist_ok=True)

output_file = output_dir / f"results_mu_var_{demography}.txt"

# Filter MR values with more than 0 points
MR_filtered = MR_counts[MR_counts > 0].index # Basically unique MR values which are non NA

with open(output_file, "w") as f:
    f.write("MR\tmean_mu\tvar_mu\tmax_loglikelihood\n")

    with ProcessPoolExecutor() as executor:
        for i, MR, mean_mu, var_mu, loglik in executor.map(optimize_for_MR, range(1,len(MR_filtered))):  # we lose the first roulette value MR = 0.004 which has 2 points
            f.write(f"{MR}\t{mean_mu}\t{var_mu}\t{loglik}\n")


Optimization terminated successfully.
         Current function value: 827840.303970
         Iterations: 66
         Function evaluations: 177
Optimization terminated successfully.
         Current function value: 319818.498398
         Iterations: 77
         Function evaluations: 192
Optimization terminated successfully.
         Current function value: 658984.201492
         Iterations: 83
         Function evaluations: 195
Optimization terminated successfully.
         Current function value: 652190.443643
         Iterations: 79
         Function evaluations: 195
Optimization terminated successfully.
         Current function value: 186070.411894
         Iterations: 80
         Function evaluations: 199
Optimization terminated successfully.
         Current function value: 915700.255665
         Iterations: 81
         Function evaluations: 211
Optimization terminated successfully.
         Current function value: 238538.577020
         Iterations: 86
         Function evaluatio

## 1 parameter model where Roulette rate is proportional to mutation rate

In [6]:
def log_likelihood_1_param(params, df_syn, n, demography = "schraiber_et_al", kmax = 5000):

    """
    Computes the negative log-likelihood for given values of mu
    - `params`: mean and var of mu
    - `df_syn`: DataFrame with allele_count and SFS counts (count)
    """
    mean_mu = float(params[0])

    # Compute log(mu_scaled) only once
    _, _, _, SFS_neutral = Gen_SFS_with_s.compute_SFS(mu = mean_mu, s = 0, n = n, kmax=kmax, demography=demography)

    # Get k values as integer array
    k_indices = df_syn["allele_count"].astype(int).values

    # Get counts values as integer array
    counts = df_syn["count"].astype(int).values

    # Compute likelihood
    log_likelihood =  np.log(SFS_neutral[k_indices] + 1e-300) * counts

    logL = np.sum(log_likelihood)  # Sum log-likelihood across all loci

    return -logL  # Return negative log-likelihood for minimization


In [7]:
# Define a function for parallel optimization
def optimize_for_MR(i):
    MR = MR_filtered[i]
    df_syn_analyze = df_syn_filtered[df_syn_filtered['MR'] == MR]
    initial_params = [MR * 5e-8]
    bounds = [(2e-10, 3e-6)]

    result = minimize(
        log_likelihood_1_param,
        initial_params,
        args=(df_syn_analyze, AN),
        bounds=bounds,
        method="Nelder-Mead",
        options={"disp": True, "fatol": 1e-6}
    )

    mean_mu = result.x[0]
    loglik = -result.fun

    return i, MR, mean_mu, loglik

# Run in parallel and write results
output_dir = Path("param_mut_rate")
output_dir.mkdir(exist_ok=True)

output_file = output_dir / f"results_mu_{demography}.txt"

# Filter MR values with more than 0 points
MR_filtered = MR_counts[MR_counts > 0].index # Basically unique MR values which are non NA

with open(output_file, "w") as f:
    f.write("MR\tmean_mu\tmax_loglikelihood\n")

    with ProcessPoolExecutor() as executor:
        for i, MR, mean_mu, loglik in executor.map(optimize_for_MR, range(1,len(MR_filtered))):  # we lose the first roulette value MR = 0.004 which has 2 points
            f.write(f"{MR}\t{mean_mu}\t{loglik}\n")


Optimization terminated successfully.
         Current function value: 186070.416330
         Iterations: 15
         Function evaluations: 30
Optimization terminated successfully.
         Current function value: 238538.582446
         Iterations: 16
         Function evaluations: 32
Optimization terminated successfully.
         Current function value: 938636.440503
         Iterations: 16
         Function evaluations: 32
Optimization terminated successfully.
         Current function value: 177863.379776
         Iterations: 16
         Function evaluations: 32
Optimization terminated successfully.
         Current function value: 915700.267350
         Iterations: 16
         Function evaluations: 32
Optimization terminated successfully.
         Current function value: 319818.505365
         Iterations: 16
         Function evaluations: 32
Optimization terminated successfully.
         Current function value: 526385.447306
         Iterations: 16
         Function evaluations: 32